In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/FMCG-Domain-Databricks/1_Setup/utilities_configuration

In [0]:
dbutils.widgets.text("catalog","fmcg","Catalog")
dbutils.widgets.text("data_source","customers","Data Source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f's3://fmcg-databricks-ashit/{data_source}/*.csv'
print(base_path)

In [0]:
df = (
    spark.read.format("csv")\
    .option("header",True)\
    .option("inferSchema",True)\
    .load(base_path)\
    .withColumn("read_timestamp",F.current_timestamp())\
    .select("*","_metadata.file_name","_metadata.file_size")
)

display(df.limit(5))

In [0]:
df.printSchema()

In [0]:
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed","true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")


##Silver Processing

In [0]:
df_bronze = spark.sql(f"select *  from {catalog}.{bronze_schema}.{data_source};")
df_bronze.show()

In [0]:
import pyspark.sql.functions as F
df_duplicates = df_bronze.groupBy("customer_id").count().filter(F.col("count")>1)

display(df_duplicates)

In [0]:
print("Rows before dropped: ",df_bronze.count())
df_silver = df_bronze.dropDuplicates(["customer_id"])
print("Rows after dropped: ",df_silver.count())

In [0]:
display(df_silver.filter(F.col("customer_name") != F.trim(F.col("customer_name")) ))

In [0]:
#trim()  → leading + trailing
#ltrim() → leading only
#rtrim() → trailing only
df_silver = df_silver.withColumn("customer_name", F.trim(F.col("customer_name")))

In [0]:
display(df_silver.filter(F.col("customer_name") != F.trim(F.col("customer_name")) ))

In [0]:
df_silver.select("city").distinct().show()

In [0]:
#typos -> correct

city_mapping = {
    'Bengaluruu' : 'Bengaluru',
    'Bengalore' : 'Bengaluru',
    
    'Hyderabadd' : 'Hyderabad',
    'Hyderbad' : 'Hyderabad',

    'NewDelhi' : 'New Delhi',
    'NewDelhee' : 'New Delhi',
    'NewDheli' : 'New Delhi'
}

#will be creating three cities that are allowed, we talk to LOB and confirm if we find anything other than this those would be garbage

allowed =['Bengaluru','Hyderabad','New Delhi']

df_silver = (
    df_silver
             .replace(city_mapping, subset=["city"])
             .withColumn("city",
                         F.when(F.col("city").isNull(),None)
                         .when(F.col("city").isin(allowed), F.col("city"))
                         .otherwise(None)
                         )
             )

#Sanity check
df_silver.select("city").distinct().show()

In [0]:
df_silver.select("customer_name").distinct().show()

In [0]:
df_silver = df_silver.withColumn("customer_name", 
                     F.when(F.col("customer_name").isNull(),None)
                     .otherwise(F.initcap("customer_name"))
)

df_silver.select("customer_name").distinct().show()

In [0]:
df_silver.filter(F.col("city").isNull()).show(truncate=False)

In [0]:
null_customer_names = ["Sprintx Nutrition","Zenathlete Foods","Primefuel Nutrition","Recovery Lane"]

df_silver.filter(F.col("customer_name").isin(null_customer_names)).show(truncate=False)

In [0]:
#Business Confirmation Note: City corrections confirmed by business team

customer_city_fix = {
    #Sprintx Nutrition
    789403 : "New Delhi",

    #Zenathlete Foods
    789420 : "Bengaluru",

    #Primefuel Nutrition
    789521 : "Hyderabad",

    #Recovery Lane 
    789603 : "Hyderabad"
}

df_fix = spark.createDataFrame([(k,v) for k,v in customer_city_fix.items()],["customer_id","fixed_city"])

display(df_fix)

In [0]:
df_silver = (df_silver.join(df_fix, on="customer_id",how="left")
             .withColumn("city",F.coalesce("city","fixed_city") #replace null with fixed_city
                        )
             .drop("fixed_city")
             )
display(df_silver)

In [0]:
#here customer_id column is integer but in our gold table we had customer_id as string

df_silver = df_silver.withColumn("customer_id",F.col("customer_id").cast("string"))
print(df_silver.printSchema())

In [0]:
df_silver = (
    df_silver
    #build final customer column: "CustomerName-City" or "CustomerName-Unknown"
    #the reason we are doing so is bcoz in our existing gold table we dont have a separate city column and we dont want to drop the city column and rather use it in conjucture with customer name
    .withColumn("customer",
                F.concat_ws("-","customer_name",F.coalesce(F.col("city"), F.lit("Unknown"))))


#static attributes aligned with parent data model
.withColumn("market",F.lit("India"))
.withColumn("platform",F.lit("Sports Bar"))
.withColumn("channel",F.lit("Acquisition"))
)

display(df_silver.limit(5))

In [0]:
df_silver.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed","true")\
    .option("mergeSchema","true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

##Gold Processing

In [0]:
df_silver = spark.sql(f"select * from {catalog}.{silver_schema}.{data_source}")

#take required columns 

df_gold = df_silver.select("customer_id", "customer_name", "city","customer","market","platform","channel")

In [0]:
df_gold.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed","true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

In [0]:
#Now we want to merge the data available in gold.sb_dim_customers with the data in gold.dim_customers

delta_table = DeltaTable.forName(spark,"fmcg.gold.dim_customers")

df_child_customers = spark.table("fmcg.gold.sb_dim_customers").select(
    F.col("customer_id").alias("customer_code"),
    "customer",
    "market",
    "platform",
    "channel"
)

In [0]:
#Will be doing upsert operation that is Update and insert based on the condition
delta_table.alias("target").merge(
    source = df_child_customers.alias("source"),
    condition="target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()